# Barranquilla MOPAC: conservación de la energía en QM/MM (tripsina–BPTI)

Las pruebas NVE anteriores duraron 0,2–2 ps, con una trayectoria por caso y sin referencia MM, así que no
permitían afirmar nada sobre la conservación de la energía. Aquí: NVE de 10 ps con pasos de 1 fs, MOPAC en la GPU
(PME en la GPU), dos semillas, dos criterios SCF y el mismo sistema sin región QM como referencia.

| Bloque | Qué hace | Tiempo (A100, estimado) |
|---|---|---|
| Compilar | MOPAC con OpenBLAS | ~10 min |
| A. NVE | por semilla: `mm` (~3 min), `qm_tol0.01` (~25 min), `qm_tol0.001` (~30 min o más) | ~2 h con 2 semillas |
| B. Energía | tabla y gráfica de E total − E(0) | segundos |
| C. Trayectoria | visor 3D animado (py3Dmol) | segundos |

Los tiempos de A son estimaciones a partir de 0,13 s por paso. **Antes de empezar:** *Entorno de ejecución → Cambiar
tipo de entorno → GPU A100*, y luego *Ejecutar todo*. Si Colab asigna otra GPU, la celda 2 se detiene. Los resultados
se guardan en Google Drive (`MyDrive/mopac_paper_bench/RESULTS_NAME`); si la sesión se corta, vuelva a ejecutar todo
y las corridas terminadas se saltan. La última celda descarga un zip.

## 1. Configuración

In [ ]:
COMMIT = '15d20491'            # run_qmmm_md con --mm-only, --nve-every y --nve-traj
RESULTS_NAME = 'qmmm_nve_long_a100'
USE_DRIVE = True
SEEDS = [11, 12]               # [11]: la mitad del tiempo
TOLS = [0.01, 0.001]           # criterio SCF relativo (0.01: el del artículo)
DT = 1.0                       # fs
NVT_STEPS = 400
NVE_STEPS = 10000              # 10 ps con 1 fs
ENERGY_EVERY = 10              # pasos entre lecturas de la energía total
TRAJ_EVERY = 200               # pasos entre cuadros de la trayectoria (50 cuadros)
QUICK = False                  # True: 20 + 200 pasos (prueba del notebook)
REQUIRE_GPU = 'A100'           # None: aceptar cualquier GPU

## 2. GPU, CPU y carpeta de resultados

In [ ]:
import os, re, subprocess, sys, time, shutil, json, gzip
from pathlib import Path

def run(cmd, check=True, env=None, log=None):
    # Run a shell command and stream its output to the cell (and to a log file)
    print('$', cmd, flush=True)
    full = dict(os.environ); full.update(env or {})
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash', stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, env=full)
    fh = open(log, 'w') if log else None
    for line in p.stdout:
        print(line, end='', flush=True)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if check and p.returncode != 0:
        raise RuntimeError(f'exit {p.returncode}: {cmd}')
    return p.returncode

run('nvidia-smi')
gpu = subprocess.run('nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader', shell=True,
                     capture_output=True, text=True).stdout.strip()
cpu = subprocess.run("lscpu | grep 'Model name' | sed 's/.*: *//'", shell=True, capture_output=True,
                     text=True).stdout.strip()
NCPU = os.cpu_count()
print(f'GPU: {gpu}\nCPU: {cpu}, {NCPU} núcleos lógicos')
CC = gpu.split(',')[-1].strip().replace('.', '')
if REQUIRE_GPU and REQUIRE_GPU not in gpu:
    raise RuntimeError(f'Esta sesión tiene {gpu}, no una {REQUIRE_GPU}. Cambie el tipo de entorno a {REQUIRE_GPU} '
                     f'(o ponga REQUIRE_GPU = None) y vuelva a ejecutar todo.')

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = Path('/content/drive/MyDrive/mopac_paper_bench') / RESULTS_NAME
else:
    OUT = Path('/content/mopac_paper_bench') / RESULTS_NAME
OUT.mkdir(parents=True, exist_ok=True)
(OUT / 'env.json').write_text(json.dumps({'gpu': gpu, 'cpu': cpu, 'ncpu': NCPU, 'commit': COMMIT}, indent=1))
print('resultados en', OUT)

## 3. Compilar MOPAC (biblioteca) con OpenBLAS

~10 minutos.

In [ ]:
SRC, BUILD = Path('/content/mopac_gpu'), Path('/content/build')
run('apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq gfortran ninja-build '
    'libopenblas-dev liblapack-dev > /dev/null')
os.environ['PATH'] = '/usr/local/cuda/bin:' + os.environ['PATH']
shutil.rmtree(SRC, ignore_errors=True); shutil.rmtree(BUILD, ignore_errors=True)
run(f'git clone -q https://github.com/juvenalyosa/mopac_gpu.git {SRC}')
run(f'git -C {SRC} checkout -q {COMMIT}')
run(f'git -C {SRC} log -1 --format="%h %ad %s" --date=short')
head = subprocess.run(f'git -C {SRC} rev-parse HEAD', shell=True, capture_output=True, text=True).stdout.strip()
if not head.startswith(COMMIT):
    raise SystemExit(f'HEAD {head} no es COMMIT {COMMIT}')
stamp = OUT / 'commit.txt'
if stamp.exists() and not head.startswith(stamp.read_text().strip()[:8]):
    raise SystemExit(f'{OUT} tiene resultados de {stamp.read_text().strip()}; use otro RESULTS_NAME')
stamp.write_text(head + '\n')
t0 = time.time()
run(f'cmake -S {SRC} -B {BUILD} -G Ninja -DGPU=ON -DTESTS=OFF -DCMAKE_BUILD_TYPE=Release -DCUDA_ARCHS={CC} '
    f'-DENABLE_MKL=OFF -DBLA_VENDOR=OpenBLAS > /content/cmake.log 2>&1 || (tail -40 /content/cmake.log; exit 1)')
run(f'cmake --build {BUILD} --target mopac --parallel {NCPU} > /content/build.log 2>&1 '
    f'|| (grep -i -B2 -A6 error /content/build.log | head -80; exit 1)')
LIBMOPAC = next(BUILD.glob('libmopac.so*'))
print(f'compilado en {time.time() - t0:.0f} s: {LIBMOPAC}')

## 4. OpenMM y CuPy

In [ ]:
m = re.search(r'release (\d+)', subprocess.run('nvcc --version', shell=True, capture_output=True, text=True).stdout)
cupy_pkg = f'cupy-cuda{m.group(1) if m else 12}x'
other = [p for p in subprocess.run('pip list 2>/dev/null', shell=True, capture_output=True, text=True).stdout.split()
         if p.startswith('cupy-cuda') and p != cupy_pkg]
if other:
    run('pip uninstall -y -q ' + ' '.join(other), check=False)
run(f"pip install -q 'openmm[cuda12]==8.6.1' {cupy_pkg} 2>&1 | tail -2", check=False)
run(f"{sys.executable} -c \"import openmm as m; print('OpenMM', m.__version__, "
    f"[m.Platform.getPlatform(i).getName() for i in range(m.Platform.getNumPlatforms())])\"", check=False)
run(f"cd {SRC}/scripts/qmmm && {sys.executable} -c \"from qmmm_pme import gpu_available; "
    f"print('QM-PME grid on the GPU (CuPy):', gpu_available())\"", check=False)

## 5. Lanzador con tiempos por llamada

`/content/qmmm_prof.py` ejecuta `run_qmmm_md.py` sin cambiarlo y envuelve cuatro métodos para medir cada llamada:
`MopacQMMMForce.compute` (toda la fuerza QM/MM), `QMSession.step` (MOPAC), `_pme_far_potential` y
`_pme_far_gradient` (PME). El tiempo entre el final de una llamada a la fuerza y el comienzo de la siguiente es el
resto del paso (OpenMM). Los tiempos se guardan en `pytimes.json`.

Modos: `cpu` = `MOPAC_NOGPU=1`; `gpu` = `MOPAC_FORCEGPU=1`. Cada variante añade variables de entorno o opciones.
Con `MOPAC_GPU_PROFILE=1`, MOPAC escribe en su salida (`MOPAC_API_OUTPUT`) el tiempo acumulado por sección después
de cada paso; se guarda el último valor de cada sección en `sections.csv` y la salida completa comprimida.

In [ ]:
PROF = Path('/content/qmmm_prof.py')
PROF.write_text('''
import json, os, runpy, sys, time
script = sys.argv[1]
sys.argv = sys.argv[1:]
sys.path.insert(0, os.path.dirname(os.path.abspath(script)))
import openmm_mopac as om
import mopac_qmmm_api as api
T = {"t0": [], "compute": [], "mopac": [], "pme_far": [], "pme_grad": [], "stack_before": [], "stack_after": [],
     "free_mb_after": [], "preset": None}
rt = None
if os.environ.get("QMMM_STACK_LOG") or os.environ.get("QMMM_STACK_PRESET"):
    try:
        import cupy
        rt = cupy.cuda.runtime
        rt.free(rt.malloc(8))                  # create the primary context (the one MOPAC uses)
        if os.environ.get("QMMM_STACK_PRESET"):
            rt.deviceSetLimit(0, int(os.environ["QMMM_STACK_PRESET"]))   # 0: cudaLimitStackSize
            T["preset"] = rt.deviceGetLimit(0)
        print("stack limit at start:", rt.deviceGetLimit(0), flush=True)
    except Exception as e:
        print("stack log unavailable:", e, flush=True)
        rt = None
def stack_log(key):
    if rt is not None:
        T[key].append(rt.deviceGetLimit(0))
        if key == "stack_after":
            T["free_mb_after"].append(rt.memGetInfo()[0] / 2**20)
def wrap(cls, name, key, stamp=False):
    f = getattr(cls, name)
    def g(self, *a, **k):
        t = time.perf_counter()
        if stamp:
            T["t0"].append(t)
        try:
            return f(self, *a, **k)
        finally:
            T[key].append(time.perf_counter() - t)
    setattr(cls, name, g)
wrap(om.MopacQMMMForce, "compute", "compute", True)
wrap(om.MopacQMMMForce, "_pme_far_potential", "pme_far")
wrap(om.MopacQMMMForce, "_pme_far_gradient", "pme_grad")
wrap(api.QMSession, "step", "mopac")
_step = api.QMSession.step
def _logged(self, *a, **k):
    stack_log("stack_before")
    try:
        return _step(self, *a, **k)
    finally:
        stack_log("stack_after")
api.QMSession.step = _logged
try:
    runpy.run_path(script, run_name="__main__")
finally:
    out = os.environ.get("QMMM_PROF_OUT")
    if out:
        with open(out, "w") as fh:
            json.dump(T, fh)
''')

RX = {
    'system': re.compile(r'system: (\d+) atoms, (\d+) residues, box ([0-9.]+) A'),
    'qm': re.compile(r'QM region: (\d+) atoms \+ (\d+) link atoms, charge (-?\d+); long-range QM-MM electrostatics: (.+)'),
    'nvt': re.compile(r'QM/MM NVT: (\d+) steps x ([0-9.]+) fs in ([0-9.]+) s = ([0-9.]+) s/step, ([0-9.]+) ps/day'),
    'nve': re.compile(r'NVE check (\d+) steps: drift (-?[0-9.]+) kJ/mol, slope of a linear fit (-?[0-9.]+) kJ/mol/ps, '
                      r'largest step ([0-9.]+), rms step ([0-9.]+), rms deviation from the fit ([0-9.]+)'),
    'sec': re.compile(r'\[PROFILE\] MOZYME_SECTION name=(\S+) calls=(\d+) ms=([0-9.]+)'),
}
QMMM = SRC / 'scripts' / 'qmmm' / 'run_qmmm_md.py'

def py_breakdown(T, first, last):
    # medians (ms) of the per-call times over calls first..last-1 (the NVT steps)
    import numpy as np
    c_all, t0 = np.array(T['compute']), np.array(T['t0'])
    c = c_all[first:last]
    # rest of the step: from the end of one force call to the start of the next
    j = np.arange(first, min(last, len(t0) - 1))
    gap = t0[j + 1] - (t0[j] + c_all[j])
    r = {'py_compute_ms': np.median(c) * 1e3 if len(c) else None,
         'py_compute_mean_ms': np.mean(c) * 1e3 if len(c) else None,
         'py_mopac_ms': np.median(T['mopac'][first:last]) * 1e3 if T['mopac'][first:last] else None,
         'py_mopac_mean_ms': np.mean(T['mopac'][first:last]) * 1e3 if T['mopac'][first:last] else None,
         'py_pme_ms': (np.median(np.array(T['pme_far'][first:last]) + np.array(T['pme_grad'][first:last])) * 1e3
                       if T['pme_far'][first:last] else 0.0),
         'py_rest_ms': np.median(gap) * 1e3 if len(gap) else None}
    return {k: (round(float(v), 2) if v is not None else None) for k, v in r.items()}

def qmmm(exp, case, variant, pdb, opts, steps, nve=0, mm_ps=10.0, env_extra=None, seed=11):
    mode = variant.split('_')[0]
    env_extra = dict(env_extra or {})
    d = OUT / 'runs' / exp / case / variant
    cmd = (f'{sys.executable} -u {PROF} {QMMM} {pdb} --libmopac {LIBMOPAC} {opts} --platform CUDA '
           f'--steps {steps} --nve {nve} --mm-equil-ps {mm_ps} --seed {seed}')
    key = cmd + '\n' + json.dumps(env_extra, sort_keys=True)
    if (d / 'result.json').exists() and (d / 'cmd.txt').exists() and (d / 'cmd.txt').read_text() == key:
        r = json.loads((d / 'result.json').read_text())
        print(f'[{exp}/{case}] {variant}: ya hecho, {r.get("s_per_step")} s/step')
        return r
    shutil.rmtree(d, ignore_errors=True); d.mkdir(parents=True)
    (d / 'cmd.txt').write_text(key)
    api_out = Path('/tmp') / f'mopac_api_{exp}_{case}_{variant}.out'
    api_out.unlink(missing_ok=True)
    env = {k: v for k, v in os.environ.items() if k not in ('MOPAC_NOGPU', 'MOPAC_FORCEGPU')}
    env.update(OMP_NUM_THREADS=str(NCPU), OPENBLAS_NUM_THREADS=str(NCPU), MOPAC_GPU_DEBUG='1',
               MOPAC_API_OUTPUT=str(api_out), QMMM_PROF_OUT=str(d / 'pytimes.json'))
    env['MOPAC_NOGPU' if mode == 'cpu' else 'MOPAC_FORCEGPU'] = '1'
    env.update(env_extra)
    env = {k: v for k, v in env.items() if v is not None}       # None: remove the variable
    print(f'[{exp}/{case}] {variant} {env_extra}', flush=True)
    t0 = time.perf_counter()
    with open(d / 'log.txt', 'w') as fh:
        rc = subprocess.run(cmd, shell=True, cwd=d, env=env, stdout=fh, stderr=subprocess.STDOUT).returncode
    text = (d / 'log.txt').read_text(errors='ignore')
    r = {'exp': exp, 'case': case, 'variant': variant, 'mode': mode, 'env': json.dumps(env_extra), 'rc': rc,
         'total_s': round(time.perf_counter() - t0, 1)}
    if (m := RX['system'].search(text)):
        r.update(total_atoms=int(m.group(1)), box_A=float(m.group(3)))
    if (m := RX['qm'].search(text)):
        r.update(qm_atoms=int(m.group(1)), link_atoms=int(m.group(2)), qm_charge=int(m.group(3)), long_range=m.group(4).strip())
    if (m := RX['nvt'].search(text)):
        r.update(steps=int(m.group(1)), dt_fs=float(m.group(2)), md_s=float(m.group(3)),
                 s_per_step=float(m.group(4)), ps_per_day=float(m.group(5)))
    if (m := RX['nve'].search(text)):
        r.update(nve_steps=int(m.group(1)), nve_drift_kj=float(m.group(2)), nve_slope_kj_ps=float(m.group(3)),
                 nve_rms_dev_kj=float(m.group(6)))
    if (m := re.search(r'NVE: energy every (\d+) steps, ([0-9.]+) s = ([0-9.]+) s/step', text)):
        r.update(nve_s=float(m.group(2)), nve_s_per_step=float(m.group(3)))
    if (d / 'pytimes.json').exists():
        T = json.loads((d / 'pytimes.json').read_text())
        r['force_calls'] = len(T['compute'])
        r.update(py_breakdown(T, 2, 2 + steps))          # the first 2 calls: first SCF from the Lewis structure
        if T.get('stack_after'):
            sb, sa, fm = T['stack_before'], T['stack_after'], T['free_mb_after']
            r.update(stack_preset=T.get('preset'), stack_first_before=sb[0], stack_first_after=sa[0],
                     stack_last_after=sa[-1], stack_max=max(sa + sb), free_mb_first=round(fm[0]),
                     free_mb_last=round(fm[-1]), free_mb_min=round(min(fm)))
    lines = api_out.read_text(errors='ignore').splitlines() if api_out.exists() else []
    r['mozyme_gpu_lines'] = sum('[MOZYME GPU' in l for l in lines)
    r['gpu_scf_stdout_lines'] = sum('[MOZYME GPU SCF]' in l for l in text.splitlines())
    sec = {}
    for l in lines:
        if (m := RX['sec'].search(l)):
            sec[m.group(1)] = (int(m.group(2)), float(m.group(3)))
    if sec:
        with open(d / 'sections.csv', 'w') as fh:
            fh.write('section,calls,ms_total\n')
            for k, (c, ms) in sorted(sec.items(), key=lambda x: -x[1][1]):
                fh.write(f'{k},{c},{ms:.3f}\n')
    if api_out.exists():
        with open(api_out, 'rb') as fi, gzip.open(d / 'mopac_api.out.gz', 'wb') as fo:
            shutil.copyfileobj(fi, fo)
        api_out.unlink()
    (d / ('result.json' if rc == 0 and 's_per_step' in r else 'result_failed.json')).write_text(json.dumps(r, indent=1))
    print(f'    {r.get("s_per_step")} s/step; per call (ms, median): force {r.get("py_compute_ms")}, MOPAC '
          f'{r.get("py_mopac_ms")}, PME {r.get("py_pme_ms")}, rest {r.get("py_rest_ms")}; [MOZYME GPU] lines '
          f'{r["mozyme_gpu_lines"]}, rc {rc}, {r["total_s"]:.0f} s', flush=True)
    if rc != 0:
        print('\n'.join(text.splitlines()[-10:]))
    return r

def get_pdb(code):
    import urllib.request
    f = OUT / 'runs' / '_pdb' / f'{code}.pdb'
    f.parent.mkdir(parents=True, exist_ok=True)
    if not f.exists():
        urllib.request.urlretrieve(f'https://files.rcsb.org/download/{code}.pdb', f)
    return f

CHIG = SRC / 'tests' / 'data' / 'mozyme_gpu_reference' / '1UAO_opt.pdb'
ENZ = ('--qm-residues E:57:sc,E:102:sc,E:195:sc,I:15,I:16 --variants E:57=HID --padding 1.0 --ron 10 --roff 12 '
       '--distances E:195:OG-I:15:C,E:57:NE2-E:195:OG,E:57:HD1-E:102:OD2')
ENZ_PDB = get_pdb('2PTC')

## A. NVE larga, tripsina–BPTI

Para cada semilla: 100 ps de MM, 400 pasos QM/MM NVT de 1 fs y luego `NVE_STEPS` pasos NVE (Verlet) de 1 fs, con la
energía total cada `ENERGY_EVERY` pasos y un cuadro de la proteína cada `TRAJ_EVERY` pasos. Variantes:

- `mm`: el mismo sistema sin región QM (los residuos QM quedan en el campo de fuerzas): referencia de la deriva del
  integrador, las restricciones y el PME de la parte MM.
- `qm_tol0.01`: QM/MM como en el artículo (`--tolerance 0.01`).
- `qm_tol0.001`: QM/MM con un criterio SCF 10 veces más estricto.

Primero se hacen las tres variantes de la primera semilla; si la sesión se corta, lo hecho queda en Drive y al
volver a ejecutar todo se salta.

In [ ]:
def nve_opts(tol=None, mm_only=False):
    o = (f'{ENZ} --dt {DT} --nve-every {ENERGY_EVERY} --nve-traj {TRAJ_EVERY}')
    o += ' --mm-only' if mm_only else f' --tolerance {tol}'
    return o
VARIANTS = [('mm', None, True)] + [(f'qm_tol{t:g}', t, False) for t in TOLS]
res = []
for seed in SEEDS:
    for name, tol, mm_only in VARIANTS:
        res.append(qmmm('A_nve', f'trypsin_seed{seed}', f'gpu_{name}', ENZ_PDB, nve_opts(tol, mm_only),
                        20 if QUICK else NVT_STEPS, nve=200 if QUICK else NVE_STEPS,
                        mm_ps=10.0 if QUICK else 100.0, seed=seed))

## B. Energía total

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
rows = []
for f in sorted((OUT / 'runs' / 'A_nve').rglob('result*.json')):
    r = json.loads(f.read_text()); r['ok'] = f.name == 'result.json'; rows.append(r)
df = pd.DataFrame(rows)
df.to_csv(OUT / 'qmmm_nve_long.csv', index=False)
cols = ['case', 'variant', 'rc', 's_per_step', 'nve_steps', 'nve_drift_kj', 'nve_slope_kj_ps', 'nve_rms_dev_kj',
        'nve_s_per_step', 'total_s', 'ok']
display(df[[c for c in cols if c in df]])

fig, ax = plt.subplots(figsize=(9, 4.5))
for f in sorted((OUT / 'runs' / 'A_nve').rglob('nve_energy.txt')):
    e = np.loadtxt(f)
    t = np.arange(len(e)) * ENERGY_EVERY * DT / 1000.0
    ax.plot(t, e - e[0], lw=0.8, label=f'{f.parent.parent.name} {f.parent.name[4:]}')
ax.set_xlabel('tiempo NVE (ps)'); ax.set_ylabel('E total − E(0) (kJ/mol)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(OUT / 'qmmm_nve_long.png', dpi=200); plt.show()

## C. Trayectoria

Cuadros de la proteína durante la NVE (átomos pesados más todos los de la región QM). Cartón para la proteína y
barras para la región QM: His57, Asp102 y Ser195 de la tripsina (cadena E) y Lys15–Ala16 del BPTI (cadena I).
Elija la corrida en `SHOW`.

In [ ]:
run('pip install -q py3Dmol', check=False)
import py3Dmol
SHOW = ('trypsin_seed%d' % SEEDS[0], 'gpu_qm_tol%g' % TOLS[0])
traj = OUT / 'runs' / 'A_nve' / SHOW[0] / SHOW[1] / 'nve_traj.pdb'
print(traj, f'{traj.stat().st_size / 1e6:.1f} MB' if traj.exists() else 'no existe')
if traj.exists():
    v = py3Dmol.view(width=900, height=600)
    v.addModelsAsFrames(traj.read_text())
    v.setStyle({'cartoon': {'colorscheme': 'chain'}})
    qm_sel = [{'chain': 'E', 'resi': [57, 102, 195]}, {'chain': 'I', 'resi': [15, 16]}]
    for s in qm_sel:
        v.addStyle(s, {'stick': {'radius': 0.2, 'colorscheme': 'orangeCarbon'}})
    v.zoomTo({'or': qm_sel})
    v.animate({'loop': 'forward', 'interval': 80})
    v.show()

## D. Descargar

In [ ]:
tmp = Path('/content/_qmmm_nve') / RESULTS_NAME
shutil.rmtree(tmp.parent, ignore_errors=True)
tmp.mkdir(parents=True)
for f in ('qmmm_nve_long.csv', 'qmmm_nve_long.png', 'env.json', 'commit.txt'):
    if (OUT / f).exists():
        shutil.copy2(OUT / f, tmp / f)
for pat in ('*.json', 'log.txt', 'cmd.txt', 'nve_energy.txt', 'nve_traj.pdb', 'mopac_api.out.gz'):
    for f in (OUT / 'runs').rglob(pat):
        if '_pdb' in f.parts:
            continue
        dst = tmp / 'runs' / f.relative_to(OUT / 'runs')
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, dst)
z = shutil.make_archive(f'/content/{RESULTS_NAME}', 'zip', tmp.parent)
print(z, f'{os.path.getsize(z) / 1e6:.1f} MB')
from google.colab import files
files.download(z)